<a href="https://colab.research.google.com/github/adithya-prabhu-22/ml_lab_2_4NI23CI007_adithya/blob/main/FOil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle

import os

!mkdir -p ~/.kaggle

!mv kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

print('Kaggle API client configured.')

!kaggle datasets download -d uciml/iris

!unzip iris.zip

print('Iris dataset downloaded and unzipped.')

import pandas as pd

iris_df = pd.read_csv('Iris.csv')

iris_df = iris_df.drop('Id', axis=1)

for col in ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']:
    iris_df[col] = iris_df[col].apply(lambda x: f'{x:.1f}')

iris_examples = iris_df.to_dict(orient='records')

iris_target = 'Species'
iris_target_value = 'Iris-setosa'
iris_predicates = [
    'SepalLengthCm',
    'SepalWidthCm',
    'PetalLengthCm',
    'PetalWidthCm'
]

print("Iris dataset loaded and preprocessed. First 5 examples:")
display(iris_df.head())

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle API client configured.
Dataset URL: https://www.kaggle.com/datasets/uciml/iris
License(s): CC0-1.0
iris.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  iris.zip
replace Iris.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace database.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
Iris dataset downloaded and unzipped.
Iris dataset loaded and preprocessed. First 5 examples:


,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
import math

def foil_gain(rule, literal, positives, negatives):
    p0 = len(covered_examples(rule, positives))
    n0 = len(covered_examples(rule, negatives))

    new_rule = rule + [literal]

    positive_after = covered_examples(new_rule, positives)
    negative_after = covered_examples(new_rule, negatives)

    p1 = len(positive_after)
    n1 = len(negative_after)

    t = p1

    if p1 == 0 or p0 == 0:
        return float("-inf")

    if (p1 + n1) == 0:
        log_term1 = 0
    else:
        log_term1 = math.log2(p1 / (p1 + n1))

    if (p0 + n0) == 0:
        log_term2 = 0
    else:
        log_term2 = math.log2(p0 / (p0 + n0))

    gain = t * (log_term1 - log_term2)

    return gain


def covered_examples(rule, examples):
    result = []

    for example in examples:
        satisfies = True

        for feature, value in rule:
            if feature not in example or example[feature] != value:
                satisfies = False
                break

        if satisfies:
            result.append(example)

    return result


def generate_candidate_literals(rule, predicates, examples):
    candidates = []

    if not examples:
        return candidates

    for feature in predicates:
        values = set()
        for example in examples:
            if feature in example:
                values.add(example[feature])

        for value in values:
            literal = (feature, value)

            if literal not in rule:
                candidates.append(literal)

    return candidates


def learn_new_rule(positives, negatives, predicates, all_examples):
    new_rule = []

    current_negatives_covered = list(negatives)

    while True:

        candidates = generate_candidate_literals(
            new_rule,
            predicates,
            all_examples
        )

        if not candidates:
            break

        best_literal = None
        max_gain = float("-inf")

        for literal in candidates:
            current_gain = foil_gain(
                new_rule,
                literal,
                positives,
                current_negatives_covered
            )
            if current_gain > max_gain:
                max_gain = current_gain
                best_literal = literal

        if best_literal is None or max_gain == float("-inf"):
            break

        new_rule.append(best_literal)

        current_negatives_covered = covered_examples(
            new_rule,
            negatives
        )

        if not current_negatives_covered:
            break

    return new_rule

In [ ]:
def foil(examples, target, target_value, predicates):
    positives = [
        example for example in examples
        if example[target] == target_value
    ]

    negatives = [
        example for example in examples
        if example[target] != target_value
    ]

    learned_rules = []

    while positives:

        new_rule = learn_new_rule(
            positives,
            negatives,
            predicates,
            examples
        )

        if not new_rule:
            break

        learned_rules.append(new_rule)

        covered_pos = covered_examples(
            new_rule,
            positives
        )

        if not covered_pos:
            break

        positives = [
            example for example in positives
            if example not in covered_pos
        ]

    return learned_rules


def print_rules(rules, target, target_value):
    for i, rule in enumerate(rules, 1):

        if rule:
            conditions = " AND ".join(
                f"{feature} = {value}"
                for feature, value in rule
            )

            print(
                f"Rule {i}: IF {conditions} "
                f"THEN {target} = {target_value}"
            )
        else:
            print(
                f"Rule {i}: IF TRUE "
                f"THEN {target} = {target_value}"
            )

In [ ]:
iris_rules = foil(
    iris_examples,
    target=iris_target,
    target_value=iris_target_value,
    predicates=iris_predicates
)

print(f"\nLearned rules for {iris_target} = {iris_target_value}:")
print_rules(iris_rules, iris_target, iris_target_value)


Learned rules for Species = Iris-setosa:
Rule 1: IF PetalWidthCm = 0.2 THEN Species = Iris-setosa
Rule 2: IF PetalLengthCm = 1.5 THEN Species = Iris-setosa
Rule 3: IF PetalWidthCm = 0.3 THEN Species = Iris-setosa
Rule 4: IF PetalWidthCm = 0.4 THEN Species = Iris-setosa
Rule 5: IF PetalWidthCm = 0.1 THEN Species = Iris-setosa
Rule 6: IF SepalWidthCm = 3.5 THEN Species = Iris-setosa
Rule 7: IF PetalLengthCm = 1.7 THEN Species = Iris-setosa
